## TE Population Analysis — GI × 1KG

**GOAL**
- Identify TE insertions (ALU, LINE1) exclusive to Indian linguistic groups  
- Characterise per-population frequency patterns  
- Annotate functional regions (ANNOVAR ensGene only)

**DATA**
- GenomeIndia: MELT-called ALU + LINE1 filtered VCFs  
- 1KG: Merged SV VCF v8 (GRCh38), 2 504 samples  

In [34]:
import subprocess
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)

### Configuration

**What to change:**  
- `GI_GROUP_COL` — switch between `"Code"` (83 sub-populations) and `"Linguistic_Group"` (8 groups)  
- `MIN_AF_GI` — minimum allele frequency in at least one GI group for a TE to be "present"  
- `MAX_AF_KG` — maximum allele frequency allowed across all 1KG super-populations (0 = strictly absent)  
- `POS_WINDOW` — positional tolerance (bp) when matching GI sites to 1KG / ANNOVAR coordinates (default ±20 bp, every bp checked)  

In [ ]:
# PATHS 
# Change BASE if you relocate the working directory
BASE    = Path("TE_ana/data_local/")
GI_DIR  = BASE / "Filtered_VCFs"
KG_DIR  = Path("TE_ana/data_local/1kgp_data/")
OUT_DIR = BASE / "tmp"
TMP_DIR = BASE / "tmp"

for d in [OUT_DIR / "tables", OUT_DIR / "plots", TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# INPUT VCFs 
GI_VCFS = {
    "ALU":   GI_DIR / "ALU.filtered.final.vcf.gz",
    "LINE1": GI_DIR / "LINE1.filtered.final.vcf.gz",
}

KG_VCF_RAW = KG_DIR / "ALL.wgs.mergedSV.v8.20130502.svs.genotypes.GRCh38.vcf.gz"

KG_VCFS = {
    "ALU":   KG_DIR / "1kg_ALU.vcf.gz",
    "LINE1": KG_DIR / "1kg_LINE1.vcf.gz",
}

# METADATA 
GI_META_FILE  = GI_DIR / "metadata_Jan_2025_noeths.csv"

# CHANGE THIS to switch population grouping resolution
GI_GROUP_COL  = "Code"   # "Code" = 83 sub-pops | "Linguistic_Group" = 8 groups
GI_SAMPLE_COL = "FID"    # column holding sample IDs

# 1KG AF INFO fields 
# Keys = labels used in output columns; values = INFO field names in 1KG VCF
KG_POP_AF = {
    "AFR": "AFR_AF",
    "AMR": "AMR_AF",
    "EAS": "EAS_AF",
    "EUR": "EUR_AF",
    "SAS": "SAS_AF",
}

# 1KG super-population sample sizes (diploid N per super-pop) 
# Standard 1000 Genomes superpop sizes (2,504 samples total).

KG_POP_N = {
    "AFR": 661,
    "AMR": 347,
    "EAS": 504,
    "EUR": 503,
    "SAS": 489,
}

# QUALITY THRESHOLDS
MIN_ASSESS = 0     # MELT ASSESS score (0–5); 3+ has split-read support
MIN_SR     = 0     # minimum split reads per site

# SINGLETON HANDLING  
# CHANGE: set to False to strip singletons (total AC==1 across all groups)
# out of gi_af / exonic_gi / all_tes / exclusive_tes at parse time, as before.
KEEP_SINGLETONS_GI = True

# EXCLUSIVITY THRESHOLDS
# CHANGE: raise MIN_AF_GI to be more stringent (e.g. 0.05 for ≥5%)
MIN_AF_GI  = 0.01  # TE must reach ≥1% in at least one GI group
MAX_AF_KG  = 0.0   # TE must be completely absent from all 1KG populations

# Thresholds swept in the multi-threshold comparison table
MIN_AF_GI_THRESHOLDS = [0.005, 0.01, 0.05]

# POSITIONAL MATCHING 
# Increase if you want to allow sloppier coordinate matching.
POS_WINDOW = 20    # bp tolerance for matching GI <-> 1KG / ANNOVAR coordinates

# SANITY CHECK 
print("File status:")
for k, v in {**GI_VCFS, "KG_RAW": KG_VCF_RAW}.items():
    print(f"  [{'EXISTS' if v.exists() else 'MISSING'}]  {k}: {v.name}")
print(f"\nGrouping column : {GI_GROUP_COL}")
print(f"Sample ID column: {GI_SAMPLE_COL}")

File status:
  [EXISTS]  ALU: ALU.filtered.final.vcf.gz
  [EXISTS]  LINE1: LINE1.filtered.final.vcf.gz
  [EXISTS]  KG_RAW: ALL.wgs.mergedSV.v8.20130502.svs.genotypes.GRCh38.vcf.gz

Grouping column : Code
Sample ID column: FID


### Annotation file paths

Only ANNOVAR files are used for functional annotation.  
Roadmap files were removed — ANNOVAR already gives us the genomic region labels needed.

In [ ]:
# ANNOVAR ANNOTATION FILES

GANNO_DIR = Path("/gpfs/data/user/shweta_lab/data/TE/TE_discovery/Jan_May_2026/Results/GenomicAnnotation")

ANNOVAR_FILES = {
    "ALU":   GANNO_DIR / "ALU_ensGene_annovar.hg38_multianno.txt",
    "LINE1": GANNO_DIR / "LINE1_ensGene_annovar.hg38_multianno.txt",
}

# ANNOVAR functional region order (high → low impact) — used for plot ordering
ANNOVAR_FUNC_ORDER = [
    "exonic", "splicing", "UTR5", "UTR3",
    "ncRNA_exonic", "ncRNA_splicing",
    "intronic", "ncRNA_intronic",
    "upstream", "downstream",
    "intergenic"
]

print("Annotation file status:")
for k, v in ANNOVAR_FILES.items():
    print(f"  [{'EXISTS' if v.exists() else 'MISSING'}]  {k}: {v.name}")

Annotation file status:
  [EXISTS]  ALU: ALU_ensGene_annovar.hg38_multianno.txt
  [EXISTS]  LINE1: LINE1_ensGene_annovar.hg38_multianno.txt


### bcftools helper functions

Utility wrappers used throughout

In [37]:
def run_cmd(cmd, desc=""):
    """Run a shell command; stream stderr; raise on non-zero exit."""
    print(f"  → {desc or ' '.join(cmd[:4])}")
    result = subprocess.run(
        cmd, shell=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    if result.returncode != 0:
        print(f"  STDERR:\n{result.stderr[-1000:]}")
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result.stdout


def normalize_chrom(chrom):
    """Strip 'chr' prefix so chromosome strings are bare ('1', '2', ..., 'X')."""
    chrom = str(chrom)
    if chrom.startswith("chr"):
        chrom = chrom[3:]
    return chrom


def get_vcf_samples(vcf_path):
    """Return the ordered list of sample IDs as they appear in the VCF header."""
    out = run_cmd(["bcftools", "query", "-l", str(vcf_path)],
                  f"Getting samples from {Path(vcf_path).name}")
    samples = [s.strip() for s in out.strip().split("\n") if s.strip()]
    print(f"  {len(samples)} samples found")
    return samples


def bcftools_query_to_df(vcf_path, format_str, col_names, filters=None):
    """
    Run 'bcftools query' and return a DataFrame.

    Parameters
    ----------
    format_str : bcftools -f format string (without trailing newline)
    col_names  : list of column names matching the format fields
    filters    : optional bcftools -i filter expression
    """
    cmd = ["bcftools", "query"]
    if filters:
        cmd += ["-i", filters]
    cmd += ["-f", format_str + "\n", str(vcf_path)]

    out = run_cmd(cmd, f"Querying {Path(vcf_path).name}")
    if not out.strip():
        print("  WARNING: empty output")
        return pd.DataFrame(columns=col_names)

    rows = [line.split("\t") for line in out.strip().split("\n")]
    return pd.DataFrame(rows, columns=col_names)


def count_variants(vcf_path):
    """Fast variant count using 'bcftools stats'."""
    out = run_cmd(["bcftools", "stats", str(vcf_path)],
                  f"Counting variants in {Path(vcf_path).name}")
    for line in out.split("\n"):
        if line.startswith("SN") and "number of records" in line:
            return int(line.split("\t")[-1].strip())
    return None

### Metadata — load and build group → sample mappings

Children from trio families are removed before AF computation.  
`groups` dict maps group label → set of sample IDs.

In [ ]:
meta = pd.read_csv(GI_META_FILE)
print(f"Raw metadata rows : {len(meta)}")

# Diagnose NAs before any filling
print("\nNA counts per key column:")
for col in [GI_GROUP_COL, "Tribe", "Child"]:
    print(f"  {col}: {meta[col].isna().sum()} NAs")

meta["Tribe"] = meta["Tribe"].fillna("CAO")

# Build composite group label 
def make_group_label(row):
    tribe = str(row["Tribe"]).strip()
    if tribe == "CAO":
        return "CAO"
    ling = str(row[GI_GROUP_COL]).strip()
    if tribe == "Yes":
        return f"{ling}_T"     # tribal
    return f"{ling}_NT"        # non-tribal 

meta["group_label"] = meta.apply(make_group_label, axis=1)

# Remove children from trio families 
meta_founders = meta[
    meta["Child"].isna() | (meta["Child"].astype(str).str.strip() != "Yes")
].copy()

print(f"\nAfter removing children:")
print(f"  Founders + unrelated : {len(meta_founders)}")
print(f"  Children removed     : {len(meta) - len(meta_founders)}")

group_counts = meta_founders["group_label"].value_counts().sort_index()
print(f"\nGroup sizes (composite label):")
print(group_counts.to_string())
print(f"\nTotal groups : {len(group_counts)}")

# Build group → sample set dict 
groups = {
    grp: set(
        sub[GI_SAMPLE_COL]
        .astype(str)
        .str.split("_").str[0]
        .str.strip()
        .tolist()
    )
    for grp, sub in meta_founders.groupby("group_label")
}

print(f"\ngroups dict keys: {sorted(groups.keys())}")

Raw metadata rows : 9768

NA counts per key column:
  Code: 50 NAs
  Tribe: 50 NAs
  Child: 9524 NAs

After removing children:
  Founders + unrelated : 9524
  Children removed     : 244

Group sizes (composite label):
AA_EPL_1_01_T      89
AA_EPL_1_02_T      59
AA_EPL_1_03_T      89
AA_EPL_1_04_T     164
AA_NDN_1_01_T      50
AA_NER_1_01_T      72
AA_SCH_1_01_T      64
CAO                49
DR_ECP_2_01_NT     55
DR_ECP_2_02_NT    160
DR_ECP_2_03_NT    145
DR_ECP_2_04_NT    122
DR_ECP_2_05_NT     85
DR_ECP_2_06_NT    159
DR_ECP_2_07_NT     63
DR_ECP_2_08_NT    142
DR_EGH_1_01_T      33
DR_EGH_2_01_NT    112
DR_EPL_1_01_T      67
DR_EPL_1_02_T      80
DR_EPL_1_03_T      78
DR_NGH_1_01_T      72
DR_NGH_1_02_T      72
DR_SDN_2_01_NT     31
DR_SDN_2_02_NT    157
DR_SDN_2_03_NT     56
DR_WCP_2_01_NT    156
DR_WCP_2_02_NT    158
DR_WCP_2_03_NT    162
DR_WGH_1_01_T      45
DR_WGH_1_02_T      75
IE_BPV_2_01_NT    109
IE_CHR_1_01_T      53
IE_ECP_2_01_NT    137
IE_ECP_2_02_NT    165
IE_EPL_1_01_

### Extract TE-type VCFs from 1KG merged SV VCF

This step is skipped if the output files already exist — delete them to re-extract.

In [39]:
def extract_1kg_te(vcf_in, vcf_out, svtype):
    """Extract one TE family from the 1KG merged SV VCF by SVTYPE INFO field."""
    if Path(vcf_out).exists():
        n = count_variants(vcf_out)
        print(f"  {vcf_out.name} exists ({n} variants) — delete to re-extract")
        return

    print(f"\nExtracting {svtype} from 1KG...")
    run_cmd([
        "bcftools", "view",
        "-i", f'SVTYPE="{svtype}"',
        "-Oz", "-o", str(vcf_out),
        str(vcf_in)
    ], f"Filtering SVTYPE={svtype}")

    run_cmd(["bcftools", "index", "-t", str(vcf_out)], "Indexing")
    n = count_variants(vcf_out)
    print(f"  ✓ {n} {svtype} variants → {vcf_out.name}")


for te_type, out_vcf in KG_VCFS.items():
    extract_1kg_te(KG_VCF_RAW, out_vcf, te_type)

  → Counting variants in 1kg_ALU.vcf.gz
  1kg_ALU.vcf.gz exists (12743 variants) — delete to re-extract
  → Counting variants in 1kg_LINE1.vcf.gz
  1kg_LINE1.vcf.gz exists (3047 variants) — delete to re-extract


### Parse pre-computed AFs from 1KG

1KG AF values are stored in INFO fields (e.g. `AFR_AF`) — no genotype parsing needed.

In [40]:
def parse_kg_afs(vcf_path, pop_af_dict):
    """
    Extract per-super-population AFs directly from 1KG INFO fields.

    Returns a DataFrame with columns:
        chrom, pos, id, svtype, svlen, meinfo, af_global,
        AF_AFR, AF_AMR, AF_EAS, AF_EUR, AF_SAS
    """
    info_fields = list(pop_af_dict.values())    # e.g. ["AFR_AF", "AMR_AF", ...]
    format_str  = "%CHROM\t%POS\t%ID\t%INFO/SVTYPE\t%INFO/SVLEN\t%INFO/MEINFO\t%INFO/AF"
    format_str += "".join(f"\t%INFO/{f}" for f in info_fields)

    base_cols = ["chrom", "pos", "id", "svtype", "svlen", "meinfo", "af_global"]
    pop_cols  = list(pop_af_dict.keys())
    all_cols  = base_cols + pop_cols

    df = bcftools_query_to_df(vcf_path, format_str, all_cols)
    if df.empty:
        return df

    df["pos"]       = pd.to_numeric(df["pos"],       errors="coerce").astype("Int64")
    df["svlen"]     = pd.to_numeric(df["svlen"],     errors="coerce")
    df["af_global"] = pd.to_numeric(df["af_global"], errors="coerce")

    for pop in pop_cols:
        df[f"AF_{pop}"] = pd.to_numeric(df[pop], errors="coerce")
        df.drop(columns=[pop], inplace=True)

    # MEINFO format: NAME,START,END,POLARITY — pull the element name
    df["te_name"] = df["meinfo"].str.split(",").str[0]

    print(f"  Parsed {len(df)} 1KG sites from {Path(vcf_path).name}")
    af_cols = [c for c in df.columns if c.startswith("AF_")]
    print(df[af_cols].describe().round(4).to_string())
    return df


kg_af = {}
for te_type, vcf_path in KG_VCFS.items():
    print(f"\n── 1KG {te_type} ──")
    kg_af[te_type] = parse_kg_afs(vcf_path, KG_POP_AF)
    kg_af[te_type].to_csv(
        OUT_DIR / "tables" / f"1kg_{te_type}_afs.tsv", sep="\t", index=False
    )


── 1KG ALU ──
  → Querying 1kg_ALU.vcf.gz
  Parsed 12743 1KG sites from 1kg_ALU.vcf.gz
           AF_AFR      AF_AMR      AF_EAS      AF_EUR      AF_SAS
count  12743.0000  12743.0000  12743.0000  12743.0000  12743.0000
mean       0.0418      0.0354      0.0379      0.0383      0.0370
std        0.0848      0.0905      0.1027      0.0989      0.0954
min        0.0000      0.0000      0.0000      0.0000      0.0000
25%        0.0000      0.0000      0.0000      0.0000      0.0000
50%        0.0061      0.0014      0.0000      0.0000      0.0000
75%        0.0408      0.0130      0.0050      0.0106      0.0112
max        0.7806      0.8040      0.7609      0.7992      0.7904

── 1KG LINE1 ──
  → Querying 1kg_LINE1.vcf.gz
  Parsed 3047 1KG sites from 1kg_LINE1.vcf.gz
          AF_AFR     AF_AMR     AF_EAS     AF_EUR     AF_SAS
count  3047.0000  3047.0000  3047.0000  3047.0000  3047.0000
mean      0.0253     0.0222     0.0256     0.0236     0.0231
std       0.0758     0.0789     0.0936    

### Compute per-group AFs from GI genotypes

**Strategy**  
1. `bcftools query -l` → sample order in VCF  
2. `bcftools query -f` → site INFO + all GTs in one pass  
3. Map sample columns to groups via metadata  
4. Parse GT strings (`0/0`, `0/1`, `1/1`, `./.`) → alt allele counts  
5. Sum per group → AC, AN, AF  


In [41]:
# GT lookup tables: genotype string → alt allele count / allele number
GT_TO_AC = {
    "0/0": 0, "0/1": 1, "1/0": 1, "1/1": 2,
    "0|0": 0, "0|1": 1, "1|0": 1, "1|1": 2,
    "./." : 0, "."  : 0, ".|.": 0,
}
GT_TO_AN = {
    "0/0": 2, "0/1": 2, "1/0": 2, "1/1": 2,
    "0|0": 2, "0|1": 2, "1|0": 2, "1|1": 2,
    "./." : 0, "."  : 0, ".|.": 0,
}


def compute_group_afs_from_gt_df(gt_df, vcf_samples, groups):
    """
    Vectorised AF / AC / AN computation per group.

    Parameters
    ----------
    gt_df       : DataFrame (n_sites × n_samples), values are GT strings
    vcf_samples : list of sample IDs in same order as gt_df columns
    groups      : dict { group_label → set of sample IDs }

    Returns
    -------
    DataFrame with AF_<group>, AC_<group>, AN_<group> columns.
    """
    # Map VCF sample IDs to column indices.
    # VCF samples may have format FID_IID; we only use FID for matching.
    sample_to_col = {s.split("_")[0]: i for i, s in enumerate(vcf_samples)}

    print("  Parsing genotypes (vectorised)...")
    gt_vals = gt_df.values   # shape: (n_sites, n_samples)

    vec_ac = np.vectorize(lambda g: GT_TO_AC.get(g, 0))
    vec_an = np.vectorize(lambda g: GT_TO_AN.get(g, 0))

    ac_mat = vec_ac(gt_vals)   # (n_sites, n_samples)
    an_mat = vec_an(gt_vals)

    results = {}
    for grp, sample_set in groups.items():
        idxs   = [sample_to_col[s] for s in sample_set if s in sample_to_col]
        n_found = len(idxs)
        n_meta  = len(sample_set)

        if n_found == 0:
            print(f"  WARNING: {grp} — 0 of {n_meta} samples found in VCF")
            results[f"AC_{grp}"] = 0
            results[f"AN_{grp}"] = 0
            results[f"AF_{grp}"] = np.nan
            continue

        if n_found < n_meta:
            print(f"  [{grp}] {n_found}/{n_meta} samples in VCF")

        ac_grp = ac_mat[:, idxs].sum(axis=1)
        an_grp = an_mat[:, idxs].sum(axis=1)
        af_grp = np.where(an_grp > 0, ac_grp / an_grp, np.nan)

        results[f"AC_{grp}"] = ac_grp
        results[f"AN_{grp}"] = an_grp
        results[f"AF_{grp}"] = af_grp

    return pd.DataFrame(results)

In [42]:
import re

def compute_group_carrier_counts(gt_df, vcf_samples, groups):
    """
    For each group count per-site HET (one ALT allele) and HOM (two ALT alleles) carriers.

    Returns a DataFrame with HET_<group> and HOM_<group> columns.
    """
    sample_idx = {s: i for i, s in enumerate(vcf_samples)}
    records = []

    for _, row in gt_df.iterrows():
        site = {}
        for grp_name, grp_samples in groups.items():
            valid = [s for s in grp_samples if s in sample_idx]
            gts   = row[valid]

            het_count = hom_count = 0
            for gt in gts:
                alleles = re.split(r"[/|]", str(gt).strip())
                if len(alleles) != 2 or "." in alleles:
                    continue
                alt_count = sum(1 for a in alleles if a not in ("0", "."))
                if alt_count == 1:
                    het_count += 1
                elif alt_count == 2:
                    hom_count += 1

            site[f"HET_{grp_name}"] = het_count
            site[f"HOM_{grp_name}"] = hom_count

        records.append(site)

    return pd.DataFrame(records)


def parse_gi_vcf(vcf_path, groups,
                 min_assess=MIN_ASSESS, min_sr=MIN_SR,
                 keep_singletons=KEEP_SINGLETONS_GI):
    """
    Full pipeline: bcftools query → parse site info + GTs → compute group AFs.

    Returns one DataFrame with:
        site metadata | AF_<group> | AC_<group> | AN_<group> | HET_<group> | HOM_<group>

    Singletons (total AC == 1 across ALL groups) are removed here UNLESS
    keep_singletons=True (controlled by KEEP_SINGLETONS_GI in Configuration).
    """
    print(f"\nProcessing {Path(vcf_path).name}")

    # ── 1. Sample order ───────────────────────────────────────────────────
    vcf_samples = get_vcf_samples(vcf_path)

    # ── 2. Site metadata with quality filter ──────────────────────────────
    # CHANGE MIN_ASSESS / MIN_SR in the Configuration cell to tighten QC
    filter_expr = f"INFO/ASSESS>={min_assess} & INFO/SR>={min_sr}"

    site_format = (
        "%CHROM\t%POS\t%ID\t"
        "%INFO/ASSESS\t%INFO/SR\t%INFO/SVLEN\t"
        "%INFO/MEINFO\t%INFO/INTERNAL\t%INFO/TSD"
    )
    site_cols = ["chrom", "pos", "id", "assess", "sr", "svlen",
                 "meinfo", "internal", "tsd"]

    print("  Querying site metadata (with quality filter)...")
    site_df = bcftools_query_to_df(
        vcf_path, site_format, site_cols, filters=filter_expr
    )

    if site_df.empty:
        print("  No variants passed QC filters")
        return pd.DataFrame()

    site_df["pos"]    = pd.to_numeric(site_df["pos"],    errors="coerce").astype("Int64")
    site_df["svlen"]  = pd.to_numeric(site_df["svlen"],  errors="coerce")
    site_df["assess"] = pd.to_numeric(site_df["assess"], errors="coerce")
    site_df["sr"]     = pd.to_numeric(site_df["sr"],     errors="coerce")

    # TE name from MEINFO: NAME,START,END,POLARITY
    site_df["te_name"] = site_df["meinfo"].str.split(",").str[0]

    # Parse INTERNAL: "GENE,LOCATION"
    internal_split      = site_df["internal"].str.split(",", n=1, expand=True)
    site_df["gene"]     = internal_split[0].fillna(".").str.strip()
    site_df["location"] = (
        internal_split[1].fillna(".").str.strip() if 1 in internal_split else "."
    )
    # Normalise EXON_1, EXON_2 → EXON
    site_df["location"] = site_df["location"].str.replace(
        r"EXON_\d+", "EXON", regex=True
    )

    print(f"  {len(site_df)} sites passed QC (ASSESS≥{min_assess}, SR≥{min_sr})")

    # ── 3. Genotypes for QC-passing sites ─────────────────────────────────
    gt_format = "[\t%GT]"
    cmd = [
        "bcftools", "query",
        "-i", filter_expr,
        "-f", gt_format + "\n",
        str(vcf_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"bcftools query failed:\n{result.stderr}")

    gt_lines = [
        line.lstrip("\t").split("\t")
        for line in result.stdout.strip().split("\n")
        if line.strip()
    ]
    gt_df = pd.DataFrame(gt_lines, columns=vcf_samples)

    if len(gt_df) != len(site_df):
        print(f"  WARNING: site rows ({len(site_df)}) ≠ GT rows ({len(gt_df)}). Trimming.")
        min_rows = min(len(gt_df), len(site_df))
        gt_df    = gt_df.iloc[:min_rows].reset_index(drop=True)
        site_df  = site_df.iloc[:min_rows].reset_index(drop=True)

    # ── 4. Per-group AFs and carrier counts ───────────────────────────────
    af_df      = compute_group_afs_from_gt_df(gt_df, vcf_samples, groups)
    carrier_df = compute_group_carrier_counts(gt_df, vcf_samples, groups)

    # ── 5. Combine ────────────────────────────────────────────────────────
    result_df = pd.concat(
        [site_df.reset_index(drop=True),
         af_df.reset_index(drop=True),
         carrier_df.reset_index(drop=True)],
        axis=1
    )

    # ── 6. Remove singletons (total AC == 1 across all groups), unless
    #      keep_singletons=True (KEEP_SINGLETONS_GI in Configuration) ────────
    # A singleton appears in exactly one chromosome copy across the whole cohort.
    ac_cols = [c for c in result_df.columns if c.startswith("AC_")]
    if not ac_cols:
        print("  WARNING: No AC_ columns found — singleton filter skipped")
    elif keep_singletons:
        total_ac     = result_df[ac_cols].sum(axis=1)
        n_singletons = (total_ac == 1).sum()
        print(f"  keep_singletons=True — retaining {n_singletons} singleton sites "
              f"(of {len(result_df)} total)")
    else:
        total_ac      = result_df[ac_cols].sum(axis=1)
        n_before      = len(result_df)
        n_singletons  = (total_ac == 1).sum()
        result_df     = result_df[total_ac != 1].copy().reset_index(drop=True)
        print(f"  Removed {n_singletons} singletons from {n_before} sites")
        print(f"  Remaining after singleton filter: {len(result_df)} sites")

    af_cols  = [c for c in result_df.columns if c.startswith("AF_")]
    het_cols = [c for c in result_df.columns if c.startswith("HET_")]
    hom_cols = [c for c in result_df.columns if c.startswith("HOM_")]
    print(f"\n  AF summary across {len(af_cols)} groups:")
    print(result_df[af_cols].describe().round(4).to_string())
    return result_df


# Run
# Genotype parsing (bcftools query over all samples + vectorised GT parsing) is
# the slowest step in this notebook. If a cached gi_{te_type}_afs.tsv already
# exists from a previous run, load it instead of re-parsing the VCF.
# CHANGE: delete the relevant tsv (or set FORCE_REPARSE_GI = True) to re-parse.
FORCE_REPARSE_GI = True

gi_af = {}
for te_type, vcf_path in GI_VCFS.items():
    cache_path = OUT_DIR / "tables" / f"gi_{te_type}_afs.tsv"

    if cache_path.exists() and not FORCE_REPARSE_GI:
        print(f"\n{cache_path.name} exists — loading cached genotype-derived AFs "
              f"(delete file or set FORCE_REPARSE_GI=True to re-parse)")
        gi_af[te_type] = pd.read_csv(cache_path, sep="\t", low_memory=False)
        print(f"  Loaded {len(gi_af[te_type])} sites, {gi_af[te_type].shape[1]} columns")
    else:
        gi_af[te_type] = parse_gi_vcf(vcf_path, groups)
        gi_af[te_type].to_csv(cache_path, sep="\t", index=False)
        print(f"\n  Saved → {cache_path.name}")


Processing ALU.filtered.final.vcf.gz
  → Getting samples from ALU.filtered.final.vcf.gz
  7478 samples found
  Querying site metadata (with quality filter)...
  → Querying ALU.filtered.final.vcf.gz
  26977 sites passed QC (ASSESS≥0, SR≥0)
  Parsing genotypes (vectorised)...
  [AA_EPL_1_02_T] 33/59 samples in VCF
  [AA_EPL_1_03_T] 88/89 samples in VCF
  [AA_EPL_1_04_T] 160/164 samples in VCF
  [AA_NDN_1_01_T] 24/50 samples in VCF
  [AA_NER_1_01_T] 66/72 samples in VCF
  [CAO] 39/49 samples in VCF
  [DR_ECP_2_01_NT] 31/55 samples in VCF
  [DR_ECP_2_02_NT] 158/160 samples in VCF
  [DR_ECP_2_03_NT] 92/145 samples in VCF
  [DR_ECP_2_04_NT] 57/122 samples in VCF
  [DR_ECP_2_05_NT] 37/85 samples in VCF
  [DR_ECP_2_07_NT] 28/63 samples in VCF
  [DR_ECP_2_08_NT] 140/142 samples in VCF
  [DR_EGH_1_01_T] 18/33 samples in VCF
  [DR_EGH_2_01_NT] 71/112 samples in VCF
  [DR_EPL_1_02_T] 77/80 samples in VCF
  [DR_SDN_2_01_NT] 30/31 samples in VCF
  [DR_SDN_2_02_NT] 156/157 samples in VCF
  [DR_SDN_2


  Saved → gi_ALU_afs.tsv

Processing LINE1.filtered.final.vcf.gz
  → Getting samples from LINE1.filtered.final.vcf.gz
  7478 samples found
  Querying site metadata (with quality filter)...
  → Querying LINE1.filtered.final.vcf.gz
  7075 sites passed QC (ASSESS≥0, SR≥0)
  Parsing genotypes (vectorised)...
  [AA_EPL_1_02_T] 33/59 samples in VCF
  [AA_EPL_1_03_T] 88/89 samples in VCF
  [AA_EPL_1_04_T] 160/164 samples in VCF
  [AA_NDN_1_01_T] 24/50 samples in VCF
  [AA_NER_1_01_T] 66/72 samples in VCF
  [CAO] 39/49 samples in VCF
  [DR_ECP_2_01_NT] 31/55 samples in VCF
  [DR_ECP_2_02_NT] 158/160 samples in VCF
  [DR_ECP_2_03_NT] 92/145 samples in VCF
  [DR_ECP_2_04_NT] 57/122 samples in VCF
  [DR_ECP_2_05_NT] 37/85 samples in VCF
  [DR_ECP_2_07_NT] 28/63 samples in VCF
  [DR_ECP_2_08_NT] 140/142 samples in VCF
  [DR_EGH_1_01_T] 18/33 samples in VCF
  [DR_EGH_2_01_NT] 71/112 samples in VCF
  [DR_EPL_1_02_T] 77/80 samples in VCF
  [DR_SDN_2_01_NT] 30/31 samples in VCF
  [DR_SDN_2_02_NT] 156


  Saved → gi_LINE1_afs.tsv


### Non-singleton filter for exonic TEs

In [43]:
def filter_non_singletons(df, min_ac=2):
    """
    Remove sites where total AC across all groups is < min_ac.

    Parameters
    ----------
    df     : DataFrame with AC_<group> columns (output of parse_gi_vcf
             or any downstream filtered subset)
    min_ac : minimum total allele count to retain (default 2 = remove singletons)
             CHANGE to 3+ if you also want to remove doubletons, etc.

    Returns
    -------
    Filtered DataFrame and a summary dict.
    """
    ac_cols = [c for c in df.columns if c.startswith("AC_")]
    if not ac_cols:
        print("WARNING: no AC_ columns found — returning df unchanged")
        return df, {}

    total_ac  = df[ac_cols].sum(axis=1)
    keep_mask = total_ac >= min_ac

    n_before  = len(df)
    n_removed = (~keep_mask).sum()
    df_out    = df[keep_mask].copy().reset_index(drop=True)

    summary = {
        "n_before":   n_before,
        "n_removed":  int(n_removed),
        "n_after":    len(df_out),
        "min_ac_used": min_ac,
    }

    print(f"  Singleton filter (min AC={min_ac}):")
    print(f"    Before : {n_before}")
    print(f"    Removed: {n_removed}  (AC < {min_ac})")
    print(f"    After  : {len(df_out)}")
    return df_out, summary

### Annotate GI TEs with ANNOVAR, then filter to exonic sites (exonic-first workflow)

In [ ]:
def load_annovar(annovar_path):
    """
    Load an ANNOVAR hg38_multianno.txt file into a positional lookup dict.

    Returns
    -------
    dict[(chrom_no_chr, pos_int)] -> {"annovar_func": str, "annovar_gene": str}

    Multiple transcripts at the same coordinate are merged with ';' separator.
    Column names are detected dynamically so the function works with any
    ANNOVAR gene model (ensGene, refGene, knownGene).
    """
    print(f"  Loading ANNOVAR: {Path(annovar_path).name}")

    df = pd.read_csv(annovar_path, sep="\t", low_memory=False)

    chr_col = "Chr"
    pos_col = "Start"   # ANNOVAR Start column is 1-based

    # Auto-detect Func and Gene columns for the gene model in use
    # CHANGE: update the regex patterns here if you switch gene model
    func_col = [c for c in df.columns if "Func" in c and "ensGene" in c][0]
    gene_col = [c for c in df.columns
                if "Gene" in c and "ensGene" in c
                and "Detail" not in c and "Exonic" not in c][0]

    # Strip 'chr' prefix so keys match the bare-chrom convention used elsewhere
    df["_chrom"] = df[chr_col].astype(str).str.replace("^chr", "", regex=True)
    df["_pos"]   = pd.to_numeric(df[pos_col], errors="coerce")
    df = df.dropna(subset=["_pos"])
    df["_pos"] = df["_pos"].astype(int)

    # Build lookup; merge multiple transcript annotations at same coordinate
    lookup = {}
    for _, row in df.iterrows():
        key  = (row["_chrom"], row["_pos"])
        func = str(row[func_col]).strip()
        gene = str(row[gene_col]).strip()
        if key not in lookup:
            lookup[key] = {"funcs": set(), "genes": set()}
        lookup[key]["funcs"].add(func)
        if gene != ".":
            lookup[key]["genes"].add(gene)

    final_lookup = {
        key: {
            "annovar_func": ";".join(sorted(val["funcs"])),
            "annovar_gene": ";".join(sorted(val["genes"])) if val["genes"] else ".",
        }
        for key, val in lookup.items()
    }

    print(f"  Loaded {len(final_lookup):,} unique ANNOVAR coordinates")
    return final_lookup


def annotate_with_annovar(te_df, annovar_lookup):
    """
    Annotate any GI TE DataFrame (chrom/pos columns) with ANNOVAR functional labels.

    """
    te_df = te_df.copy()

    annovar_funcs = []
    annovar_genes = []
    ann_hits      = 0

    for _, row in te_df.iterrows():
        chrom = str(row["chrom"]).replace("chr", "")
        pos   = int(row["pos"])
        av_hit = annovar_lookup.get((chrom, pos))

        if av_hit:
            ann_hits += 1
            annovar_funcs.append(av_hit["annovar_func"])
            annovar_genes.append(av_hit["annovar_gene"])
        else:
            annovar_funcs.append("Unannotated")
            annovar_genes.append(".")

    te_df["annovar_func"] = annovar_funcs
    te_df["annovar_gene"] = annovar_genes

    print(f"  ANNOVAR matches (exact) : {ann_hits:,}/{len(te_df):,}")
    return te_df


# ── Annotate ALL QC-passed GI sites, then keep only exonic / ncRNA_exonic ──
annotated_gi = {}   # every QC-passed GI site, with ANNOVAR annotation
exonic_gi    = {}   # subset restricted to exonic sites -- exclusivity is checked on THIS

for te_type, df in gi_af.items():
    print(f"\n{chr(9552)*60}")
    print(f"Annotating {te_type} (all QC-passed sites, before exclusivity check)")
    print(f"{chr(9552)*60}")

    annovar_lookup = load_annovar(ANNOVAR_FILES[te_type])
    annotated_gi[te_type] = annotate_with_annovar(df, annovar_lookup)

    # Match "exonic". annovar_func can hold multiple
    # ';'-joined labels (e.g. "exonic;splicing") when several transcripts overlap
    # the same coordinate, so this checks for an exact "exonic" token rather
    # than a substring (a plain .str.contains("exonic") would also catch
    # "ncRNA_exonic", since it contains "exonic" as a substring).
    # CHANGE: add more exact labels here (e.g. "splicing") to widen the set.
    def _is_exact_exonic(func_str):
        tokens = [t.strip().lower() for t in str(func_str).split(";")]
        return "exonic" in tokens

    exonic_mask = annotated_gi[te_type]["annovar_func"].apply(_is_exact_exonic)
    exonic_gi[te_type] = annotated_gi[te_type][exonic_mask].copy().reset_index(drop=True)

    print(f"\n  {te_type}: {len(df)} total QC-passed sites -> {len(exonic_gi[te_type])} exonic sites")
    print(exonic_gi[te_type]["annovar_func"].value_counts(dropna=False).to_string())

    # Save ALL exonic GI sites (before the exclusivity check) -- this is every
    # exonic TE call in GI, not just the ones that turn out to be GI-exclusive.
    exonic_out_path = OUT_DIR / "tables" / f"{te_type}_exonic_GI_all.tsv"
    exonic_gi[te_type].to_csv(exonic_out_path, sep="\t", index=False)
    print(f"  Saved -> {exonic_out_path.name}")



════════════════════════════════════════════════════════════
Annotating ALU (all QC-passed sites, before exclusivity check)
════════════════════════════════════════════════════════════
  Loading ANNOVAR: ALU_ensGene_annovar.hg38_multianno.txt
  Loaded 26,993 unique ANNOVAR coordinates
  ANNOVAR matches (exact) : 26,977/26,977

  ALU: 26977 total QC-passed sites -> 58 exonic sites
exonic    58
  Saved -> ALU_exonic_GI_all.tsv

════════════════════════════════════════════════════════════
Annotating LINE1 (all QC-passed sites, before exclusivity check)
════════════════════════════════════════════════════════════
  Loading ANNOVAR: LINE1_ensGene_annovar.hg38_multianno.txt
  Loaded 7,094 unique ANNOVAR coordinates
  ANNOVAR matches (exact) : 7,075/7,075

  LINE1: 7075 total QC-passed sites -> 11 exonic sites
exonic    11
  Saved -> LINE1_exonic_GI_all.tsv


### Identify GI-exclusive TEs within the exonic set

**Criteria** (now applied only to the exonic sites identified above)
- AF ≥ `MIN_AF_GI` in at least one GI group
- AF ≤ `MAX_AF_KG` (default 0) in ALL 1KG super-populations
- Coordinate matching: chromosome + position within ±`POS_WINDOW` bp (default
  20 bp) — **every bp offset in that window is checked**, not a sampled subset
- Sites absent from 1KG entirely → treated as AF = 0 in all 1KG populations

In [45]:
def find_exclusive_tes(gi_df, kg_df,
                       min_af_gi=MIN_AF_GI,
                       max_af_kg=MAX_AF_KG,
                       pos_window=POS_WINDOW):
    """
    Find TEs present in GI (exonic set) but absent / rare in all 1KG super-populations.

    Coordinate matching checks EVERY bp offset within +/-pos_window (exact
    position first, then outward one bp at a time) rather than exact-only
    matching or a sampled subset of offsets.

    Returns (full_gi_df_annotated, exclusive_subset_df).
    """
    gi_af_cols = [c for c in gi_df.columns if c.startswith("AF_")]
    kg_af_cols = [c for c in kg_df.columns if c.startswith("AF_")]

    # Build 1KG position index: (chrom, pos) -> {max_af, pos, af_map}
    # af_map keeps the per-population AF values so a matched site's carrier
    # counts can be broken out by population later, not just the max AF.
    print("  Building 1KG position index...")
    kg_index = {}
    for _, row in kg_df.iterrows():
        chrom  = normalize_chrom(row["chrom"])
        pos    = int(row["pos"]) if pd.notna(row["pos"]) else -1
        af_map = {c: float(row[c]) for c in kg_af_cols if pd.notna(row[c])}
        max_af = max(af_map.values()) if af_map else 0.0
        kg_index[(chrom, pos)] = {"max_af": max_af, "pos": pos, "af_map": af_map}

    # Look up 1KG AF for each GI site, checking every bp within +/-pos_window
    print(f"  Matching GI sites to 1KG (+/-{pos_window} bp, every bp checked)...")
    max_kg_afs    = []
    matched_kg_pos = []
    matched_af_maps = []
    for _, row in gi_df.iterrows():
        chrom = normalize_chrom(row["chrom"])
        pos   = int(row["pos"]) if pd.notna(row["pos"]) else -1
        found, found_pos, found_af_map = 0.0, None, {}
        for offset in range(0, pos_window + 1):
            key = (chrom, pos + offset)
            if key in kg_index:
                entry = kg_index[key]
                found, found_pos, found_af_map = entry["max_af"], entry["pos"], entry["af_map"]
                break
            if offset != 0:
                key = (chrom, pos - offset)
                if key in kg_index:
                    entry = kg_index[key]
                    found, found_pos, found_af_map = entry["max_af"], entry["pos"], entry["af_map"]
                    break
        max_kg_afs.append(found)
        matched_kg_pos.append(found_pos)
        matched_af_maps.append(found_af_map)

    gi_df = gi_df.copy()
    gi_df["max_af_kg"]     = max_kg_afs
    gi_df["matched_kg_pos"] = matched_kg_pos
    gi_df["in_1kg"]        = gi_df["max_af_kg"] > 0
    gi_df["max_af_gi"]     = gi_df[gi_af_cols].max(axis=1)
    gi_df["is_exclusive"]  = (
        (gi_df["max_af_gi"] >= min_af_gi) &
        (gi_df["max_af_kg"] <= max_af_kg)
    )

    # Attach per-population 1KG AFs (kg_AF_<pop>) for matched sites, so
    # carrier counts can be broken out by population downstream.
    kg_af_map_df = pd.DataFrame(
        [{f"kg_{c}": m.get(c, 0.0) for c in kg_af_cols} for m in matched_af_maps],
        index=gi_df.index
    )
    gi_df = pd.concat([gi_df, kg_af_map_df], axis=1)

    exclusive = gi_df[gi_df["is_exclusive"]].copy()

    n_total   = len(gi_df)
    n_present = (gi_df["max_af_gi"] >= min_af_gi).sum()
    n_in_1kg  = gi_df["in_1kg"].sum()
    n_excl    = len(exclusive)

    print(f"\n  Total exonic GI sites                    : {n_total}")
    print(f"  Present in >=1 GI group (AF>={min_af_gi}) : {n_present}")
    print(f"  Of those, found in 1KG (+/-{pos_window}bp) : {n_in_1kg}")
    print(f"  GI-exclusive (AF<={max_af_kg} in 1KG)      : {n_excl}")

    return gi_df, exclusive


exclusive_tes = {}   # final: exonic AND GI-exclusive (pre non-singleton filter)
all_tes       = {}   # all exonic GI sites, with 1KG lookup result attached

for te_type in GI_VCFS.keys():
    print(f"\n{chr(9552)*2} {te_type} {chr(9552)*40}")
    all_df, excl_df = find_exclusive_tes(exonic_gi[te_type], kg_af[te_type])
    all_tes[te_type]       = all_df
    exclusive_tes[te_type] = excl_df

    excl_df.to_csv(
        OUT_DIR / "tables" / f"{te_type}_GI_exclusive.tsv",
        sep="\t", index=False
    )
    print(f"  Saved -> {te_type}_GI_exclusive.tsv")


# ── Multi-threshold comparison (within the exonic set) ─────────────────────
# MAX_AF_KG fixed at 0; MIN_AF_GI swept across [0.005, 0.01, 0.05]
# CHANGE MIN_AF_GI_THRESHOLDS in Configuration to sweep different values
print("\n" + chr(9552)*60)
print("MULTI-THRESHOLD COMPARISON  (exonic set, MAX_AF_KG=0, varying MIN_AF_GI)")
print(chr(9552)*60)

comparison_rows = []
multi_threshold_exclusive = {}

for te_type in GI_VCFS.keys():
    multi_threshold_exclusive[te_type] = {}
    for thresh in MIN_AF_GI_THRESHOLDS:
        print(f"\n  {te_type}  |  MIN_AF_GI={thresh}  MAX_AF_KG=0")
        _, excl_thresh = find_exclusive_tes(
            exonic_gi[te_type], kg_af[te_type],
            min_af_gi=thresh, max_af_kg=0.0
        )
        multi_threshold_exclusive[te_type][thresh] = excl_thresh

        af_cols_gi = [c for c in excl_thresh.columns if c.startswith("AF_")]
        n_grps_with_excl = sum(
            1 for col in af_cols_gi
            if (excl_thresh[col] >= thresh).any()
        ) if not excl_thresh.empty else 0

        comparison_rows.append({
            "TE_type":          te_type,
            "min_AF_GI":        thresh,
            "max_AF_KG":        0.0,
            "n_exclusive":      len(excl_thresh),
            "n_groups_with_TE": n_grps_with_excl,
            "n_total_exonic_GI_sites": len(exonic_gi[te_type]),
            "pct_exclusive":    round(100 * len(excl_thresh) / max(len(exonic_gi[te_type]), 1), 2),
        })

comparison_df = pd.DataFrame(comparison_rows)
print("\n-- Threshold comparison table --")
print(comparison_df.to_string(index=False))
comparison_df.to_csv(OUT_DIR / "tables" / "threshold_comparison.tsv", sep="\t", index=False)



══ ALU ════════════════════════════════════════
  Building 1KG position index...
  Matching GI sites to 1KG (+/-20 bp, every bp checked)...

  Total exonic GI sites                    : 58
  Present in >=1 GI group (AF>=0.01) : 14
  Of those, found in 1KG (+/-20bp) : 2
  GI-exclusive (AF<=0.0 in 1KG)      : 12
  Saved -> ALU_GI_exclusive.tsv

══ LINE1 ════════════════════════════════════════
  Building 1KG position index...
  Matching GI sites to 1KG (+/-20 bp, every bp checked)...

  Total exonic GI sites                    : 11
  Present in >=1 GI group (AF>=0.01) : 2
  Of those, found in 1KG (+/-20bp) : 0
  GI-exclusive (AF<=0.0 in 1KG)      : 2
  Saved -> LINE1_GI_exclusive.tsv

════════════════════════════════════════════════════════════
MULTI-THRESHOLD COMPARISON  (exonic set, MAX_AF_KG=0, varying MIN_AF_GI)
════════════════════════════════════════════════════════════

  ALU  |  MIN_AF_GI=0.005  MAX_AF_KG=0
  Building 1KG position index...
  Matching GI sites to 1KG (+/-20 bp, e

### Exonic vs GI-exclusive comparison table

Two-column Yes/No table over every exonic GI site: `Exonic` (always "Yes" here,
since `all_tes` is already the exonic-only set) and `GI_Exclusive` (whether
that site passed the exclusivity check against 1KG, before the non-singleton
filter below).

In [56]:
print("\n" + chr(9552)*70)
print("EXONIC vs GI-EXCLUSIVE COMPARISON TABLE")
print(chr(9552)*70)

exonic_vs_exclusive = {}

for te_type, df in all_tes.items():
    id_cols = [c for c in ["chrom", "pos", "te_name", "annovar_gene", "annovar_func"]
               if c in df.columns]

    comp_df = df[id_cols].copy()
    comp_df["Exonic"]       = "Yes"   # all_tes is already restricted to exonic sites
    comp_df["GI_Exclusive"] = df["is_exclusive"].map({True: "Yes", False: "No"})

    exonic_vs_exclusive[te_type] = comp_df

    n_yes = (comp_df["GI_Exclusive"] == "Yes").sum()
    n_no  = (comp_df["GI_Exclusive"] == "No").sum()
    print(f"\n{te_type}: {len(comp_df)} exonic sites total  |  "
          f"GI_Exclusive=Yes: {n_yes}  |  GI_Exclusive=No: {n_no}")

    out_path = OUT_DIR / "tables" / f"{te_type}_exonic_vs_exclusive_comparison.tsv"
    comp_df.to_csv(out_path, sep="\t", index=False)
    df[(df[[c for c in df.columns if c.startswith("AC_")]].sum(axis=1) == 1) & (~df["is_exclusive"])].to_csv(OUT_DIR / "tables" / f"{te_type}_singleton_nonexclusive_exonic.tsv", sep="\t", index=False)
    print(f"  Saved -> {out_path.name}")



══════════════════════════════════════════════════════════════════════
EXONIC vs GI-EXCLUSIVE COMPARISON TABLE
══════════════════════════════════════════════════════════════════════

ALU: 58 exonic sites total  |  GI_Exclusive=Yes: 12  |  GI_Exclusive=No: 46
  Saved -> ALU_exonic_vs_exclusive_comparison.tsv

LINE1: 11 exonic sites total  |  GI_Exclusive=Yes: 2  |  GI_Exclusive=No: 9
  Saved -> LINE1_exonic_vs_exclusive_comparison.tsv


### 1KG carrier counts for exonic sites that overlapped within ±20 bp

For exonic GI sites that WERE found in 1KG within the ±20 bp window (`in_1kg == True`,
i.e. the ones excluded from the exclusive set), print the population(s) with a
nonzero 1KG AF and an estimated number of carriers in that population.

**Note:** the 1KG SV VCF used here only carries pre-computed per-superpopulation
AFs (no per-sample genotypes were queried for 1KG anywhere in this notebook), so
carrier counts are estimated as `round(AF × N)` using the superpopulation sizes
in `KG_POP_N` (Configuration) — not counted directly from genotypes. Treat these
as approximate; for exact carrier counts, genotypes would need to be queried
from `KG_VCFS[te_type]` at `matched_kg_pos` for the relevant superpopulation's
sample list.

In [57]:
print("\n" + chr(9552)*70)
print("1KG CARRIERS  --  exonic GI sites overlapping 1KG within ±{}bp".format(POS_WINDOW))
print(chr(9552)*70)

kg_af_cols_by_type = {
    te_type: [c for c in df.columns if c.startswith("AF_")]
    for te_type, df in kg_af.items()
}

for te_type, df in all_tes.items():
    overlap_df = df[df["in_1kg"]].copy()
    print(f"\n{te_type}: {len(overlap_df)} exonic sites overlapped 1KG within ±{POS_WINDOW}bp")

    if overlap_df.empty:
        continue

    kg_pop_cols = [f"kg_{c}" for c in kg_af_cols_by_type[te_type]]

    for _, row in overlap_df.iterrows():
        gene = row.get("annovar_gene", ".")
        print(f"\n  {row['chrom']}:{row['pos']}  ({gene})  "
              f"matched 1KG pos {row['matched_kg_pos']}  "
              f"(offset {int(row['pos']) - int(row['matched_kg_pos'])} bp)")

        for col in kg_pop_cols:
            af = row.get(col, 0.0)
            if pd.isna(af) or af <= 0:
                continue
            pop = col.replace("kg_AF_", "")
            n_pop = KG_POP_N.get(pop)
            if n_pop is None:
                print(f"    {pop}: AF={af:.4f}  (no KG_POP_N entry -- carrier count skipped)")
                continue
            est_carriers = round(af * n_pop)
            print(f"    {pop}: AF={af:.4f}  carriers≈{est_carriers} (of {n_pop})")



══════════════════════════════════════════════════════════════════════
1KG CARRIERS  --  exonic GI sites overlapping 1KG within ±20bp
══════════════════════════════════════════════════════════════════════

ALU: 2 exonic sites overlapped 1KG within ±20bp

  chr1:12881899  (exonic)  matched 1KG pos 12881899.0  (offset 0 bp)
    AFR: AF=0.0098  carriers≈6 (of 661)
    AMR: AF=0.0086  carriers≈3 (of 347)
    EAS: AF=0.0169  carriers≈9 (of 504)
    EUR: AF=0.0169  carriers≈9 (of 503)
    SAS: AF=0.0041  carriers≈2 (of 489)

  chr18:6850866  (exonic)  matched 1KG pos 6850866.0  (offset 0 bp)
    AFR: AF=0.1241  carriers≈82 (of 661)
    AMR: AF=0.0115  carriers≈4 (of 347)

LINE1: 0 exonic sites overlapped 1KG within ±20bp


### Exonic GI-exclusive set: singleton-inclusive and non-singleton versions


In [ ]:
print("\n" + chr(9552)*70)
print("EXONIC GI-exclusive TEs  (ANNOVAR-annotated -- singleton-inclusive AND non-singleton)")
print(chr(9552)*70)

for te_type, df in exclusive_tes.items():
    n_before = len(df)

    # Save the singleton-inclusive version first (df, unfiltered) 
    with_singletons_path = OUT_DIR / "tables" / f"{te_type}_exclusive_exonic_with_singletons.tsv"
    df.to_csv(with_singletons_path, sep="\t", index=False)
    print(f"\n{te_type}: {n_before} exonic-exclusive sites (singletons included)")
    print(f"  Saved -> {with_singletons_path.name}")

    # Then the non-singleton filtered version, as before 
    # min_ac=2 removes sites with total AC=1 across all groups (singletons).
    # CHANGE min_ac to 3 if you also want to exclude doubletons.
    df_filt, filt_summary = filter_non_singletons(df, min_ac=2)
    exclusive_tes[te_type] = df_filt

    print(f"  {te_type}: {n_before} exonic-exclusive sites -> {len(df_filt)} after non-singleton filter")

    if len(df_filt):
        display_cols = [
            "chrom", "pos", "te_name",
            "annovar_gene", "annovar_func",
            "max_af_gi", "max_af_kg"
        ]
        ac_cols = [c for c in df_filt.columns if c.startswith("AC_")]
        print(df_filt[display_cols + ac_cols[:4]].to_string(index=False))
    else:
        print("  None found after filtering")

    out_path = OUT_DIR / "tables" / f"{te_type}_exclusive_exonic_nonsingletons.tsv"
    df_filt.to_csv(out_path, sep="\t", index=False)
    print(f"  Saved -> {out_path.name}")



══════════════════════════════════════════════════════════════════════
EXONIC GI-exclusive TEs  (ANNOVAR-annotated -- singleton-inclusive AND non-singleton)
══════════════════════════════════════════════════════════════════════

ALU: 12 exonic-exclusive sites (singletons included)
  Saved -> ALU_exclusive_exonic_with_singletons.tsv
  Singleton filter (min AC=2):
    Before : 12
    Removed: 4  (AC < 2)
    After  : 8
  ALU: 12 exonic-exclusive sites -> 8 after non-singleton filter
chrom       pos  te_name annovar_gene annovar_func  max_af_gi  max_af_kg  AC_AA_EPL_1_01_T  AC_AA_EPL_1_02_T  AC_AA_EPL_1_03_T  AC_AA_EPL_1_04_T
 chr1 120810467   AluYb8       exonic       exonic   0.015625        0.0                 0                 1                 0                 0
 chr5  79737711   AluYc1       exonic       exonic   0.111111        0.0                 0                 0                 0                 0
 chr8 100573918 AluYa1_2       exonic       exonic   0.020833        0.0      

### Population-wise exclusive TE counts

In [54]:
print("\n" + "═"*60)
print("POPULATION-WISE EXCLUSIVE TE COUNTS  (AF >= MIN_AF_GI in that group)")
print("═"*60)

pop_counts_all = {}

for te_type, df in exclusive_tes.items():
    if df.empty:
        print(f"  {te_type}: empty dataframe")
        continue

    af_cols   = [c for c in df.columns if c.startswith("AF_")]
    grp_names = [c.replace("AF_", "") for c in af_cols]

    rows = []
    for col, grp in zip(af_cols, grp_names):
        n_excl = (df[col] >= MIN_AF_GI).sum()
        max_af = df[col].max()
        rows.append({
            "group":             grp,
            "n_exclusive_TEs":   int(n_excl),
            "max_AF_in_group":   round(float(max_af), 4) if not pd.isna(max_af) else 0.0,
        })

    pop_df = (
        pd.DataFrame(rows)
        .sort_values("n_exclusive_TEs", ascending=False)
        .reset_index(drop=True)
    )
    pop_counts_all[te_type] = pop_df

    print(f"\n  {te_type} — exclusive TEs per population group:")
    print(pop_df.to_string(index=False))
    print(f"  Groups with ≥1 exclusive TE : {(pop_df['n_exclusive_TEs'] > 0).sum()}")
    print(f"  Total unique exclusive sites : {len(df)}")

    pop_df.to_csv(
        OUT_DIR / "tables" / f"{te_type}_population_wise_exclusive_counts.tsv",
        sep="\t", index=False
    )


# Multi-threshold pivot
print("\n" + "═"*60)
print("POPULATION-WISE COUNTS — across MIN_AF_GI thresholds")
print("═"*60)

for te_type in GI_VCFS.keys():
    thresh_rows = []
    for thresh, excl_th_df in multi_threshold_exclusive.get(te_type, {}).items():
        if excl_th_df.empty:
            continue
        af_cols = [c for c in excl_th_df.columns if c.startswith("AF_")]
        for col in af_cols:
            grp = col.replace("AF_", "")
            n   = (excl_th_df[col] >= thresh).sum()
            thresh_rows.append({"min_AF_GI": thresh, "group": grp, "n_exclusive_TEs": int(n)})

    if not thresh_rows:
        continue
    thresh_pop_df = pd.DataFrame(thresh_rows)
    pivot = thresh_pop_df.pivot_table(
        index="group", columns="min_AF_GI",
        values="n_exclusive_TEs", aggfunc="sum", fill_value=0
    )
    pivot.columns = [f"minAF_{c}" for c in pivot.columns]
    pivot = pivot.sort_values(pivot.columns[-1], ascending=False)
    print(f"\n  {te_type} — population counts by threshold:")
    print(pivot.to_string())
    pivot.to_csv(
        OUT_DIR / "tables" / f"{te_type}_population_threshold_pivot.tsv",
        sep="\t"
    )


════════════════════════════════════════════════════════════
POPULATION-WISE EXCLUSIVE TE COUNTS  (AF >= MIN_AF_GI in that group)
════════════════════════════════════════════════════════════

  ALU — exclusive TEs per population group:
         group  n_exclusive_TEs  max_AF_in_group
 DR_EGH_1_01_T                1           0.1111
 AA_EPL_1_02_T                1           0.0156
IE_EPL_2_01_NT                1           0.0122
 IE_WHR_1_01_T                1           0.0114
DR_WCP_2_03_NT                1           0.0156
DR_WCP_2_02_NT                1           0.0446
DR_SDN_2_03_NT                1           0.0189
IE_ERP_2_03_NT                1           0.0216
 DR_NGH_1_02_T                1           0.0278
 DR_NGH_1_01_T                1           0.0347
 DR_EPL_1_03_T                1           0.0256
 DR_EPL_1_02_T                1           0.0130
 DR_EPL_1_01_T                1           0.0149
IE_WCP_2_01_NT                1           0.0200
IE_WCP_2_03_NT              

### Output file inventory